In [1]:
import numpy as np

def european_call_binomial(S0: float, K: float, N: int, r: float, u: float, d: float = None) -> float:

    # 1. Set up model parameters
    if d is None:
        d = 1 / u
        
    R = 1 + r

    # 2. Calculate risk-neutral probabilities
    # This is based on the no-arbitrage condition: d < R < u
    p_u = (R - d) / (u - d)
    p_d = 1 - p_u

    # 3. Handle arbitrage case
    if not (0 <= p_u <= 1):
        raise ValueError(
            f"Arbitrage opportunity exists. Risk-neutral probability p_u = {p_u:.4f} is not in [0, 1]. "
            "Ensure that d < 1+r < u."
        )

    # 4. Initialize a 2D NumPy array for the option value lattice
    # The lattice has (N+1) time steps (0 to N).
    # At time k, there are k+1 possible states (j=0 to k up-moves).
    # We use a (N+1)x(N+1) array for simplicity.
    option_values = np.zeros((N + 1, N + 1))

    # 5. Calculate option values at maturity (time N) using backward induction
    # At time N, the option value is its intrinsic payoff: max(S_N - K, 0)
    # We calculate this for each possible final state j (number of up-moves).
    for j in range(N + 1):
        final_stock_price = S0 * (u**j) * (d**(N - j))
        option_values[N, j] = max(final_stock_price - K, 0)

    # 6. Perform backward induction to fill the rest of the lattice
    # Iterate from time N-1 down to 0
    for k in range(N - 1, -1, -1):
        # For each time k, iterate through the possible states j (0 to k up-moves)
        for j in range(k + 1):
            # Apply the risk-neutral pricing formula (recursion step)
            # v(k, j) = (1/R) * [p_u * v(k+1, j+1) + p_d * v(k+1, j)]
            value_up = option_values[k + 1, j + 1]
            value_down = option_values[k + 1, j]
            option_values[k, j] = (1 / R) * (p_u * value_up + p_d * value_down)

    # 7. The option price at time 0 is the value at the root of the lattice
    return option_values[0, 0]

# --- Test with the provided parameters ---
# Input Parameters
S0_test = 100
K_test = 105
N_test = 3
r_test = 0.05
u_test = 1.1

# Run the function and print the result
try:
    option_price = european_call_binomial(S0=S0_test, K=K_test, N=N_test, r=r_test, u=u_test)
    print(f"--- Test Case ---")
    print(f"Initial Price (S0): {S0_test}")
    print(f"Strike Price (K): {K_test}")
    print(f"Periods (N): {N_test}")
    print(f"Risk-free rate (r): {r_test}")
    print(f"Up-factor (u): {u_test}")
    print("-" * 20)
    print(f"The calculated European call option price v(0,0) is: {option_price:.4f}")
except ValueError as e:
    print(f"Error: {e}")



--- Test Case ---
Initial Price (S0): 100
Strike Price (K): 105
Periods (N): 3
Risk-free rate (r): 0.05
Up-factor (u): 1.1
--------------------
The calculated European call option price v(0,0) is: 11.6094


In [ ]:
    """
    Prices a European call option using a multi-period binomial lattice model.

    This function uses iterative backward induction (dynamic programming) without recursion.

    Args:
        S0 (float): Initial asset price.
        K (float): Strike price of the option.
        N (int): Number of periods (time steps).
        r (float): Risk-free rate per period.
        u (float): Up-factor for the asset price (u > 1).
        d (float, optional): Down-factor for the asset price. Defaults to 1/u.

    Returns:
        float: The price of the European call option at time 0.

    Raises:
        ValueError: If the risk-neutral probabilities are not between 0 and 1,
                    indicating an arbitrage opportunity in the model parameters.
    """